In [ ]:
import numpy as np
import numpy.random as npr
import numpy.linalg as npla
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

np.set_printoptions(precision=2, suppress=True)

## Polynomial features

Here is some data. We want to fit a curve to it.

In [ ]:
npr.seed(3)
n = 15
x = np.linspace(-1,1,n)
y = 9*x**2 + npr.randn(n)

plt.scatter(x,y,c='red');

A straight line won't do. Instead we can use polynomial features: for degree $p$, each row of the matrix $Z$ is $(1, x_i, x_i^2, \ldots, x_i^p)$. Then we do least squares with $Z$.

Fill in the two lines of linear algebra below. `Zgrid` is the same kind of matrix as `Z`, built from many closely spaced $x$ values so that we can draw a smooth curve.

In [ ]:
X = x[:,None]
grid = np.linspace(x.min(), x.max(), 1000)[:,None]

plt.figure(figsize=(14,6))
print("p  |  train err   |")
print("---+--------------+")
for i, p in enumerate([0, 1, 2, 4, 6, 8, 10, 12]):
    pf = PolynomialFeatures(p)
    Z = pf.fit_transform(X)
    Zgrid = pf.transform(grid)

    ############################################################
    ###                                                      ###
    ###   TODO: solve the normal equations for w             ###
    ###                                                      ###
    ############################################################
    w = 0

    ############################################################
    ###                                                      ###
    ###   TODO: predictions on the data                      ###
    ###                                                      ###
    ############################################################
    train_pred = 0

    train_err = np.sum((y-train_pred)**2)
    print('%-2d %12.2f' % (p, train_err))

    plt.subplot(2,4,i+1)
    plt.scatter(x,y,c='red',s=10)
    curve = Zgrid@w
    plt.plot(grid, curve)
    plt.title('$p=%d$'%p)
    plt.ylim((y.min()-3, y.max()+3))
plt.tight_layout();

Look at the table and the plots.

- What happens to the training error as $p$ increases?
- Which curve has the lowest error? Would you trust it to predict $y$ for a new $x$?

## Overfitting

A more complex model can always get closer to the data we fit it on. At some point it is fitting the random noise rather than the real pattern. This is called **overfitting**, and it is why the high-degree curves look so silly.

So training error can't tell us which $p$ to pick. To measure overfitting, we can **withhold** some of the data:

- **training data**: used to compute $w$
- **validation data**: hidden while computing $w$, then used to check the predictions

Let's split the same data into these two parts.

In [ ]:
xtrain,xvalid,ytrain,yvalid = train_test_split(x,y,test_size=7,random_state=3)

plt.scatter(xtrain,ytrain,c='red', label='train');
plt.scatter(xvalid,yvalid,c='green',label='validation');
plt.legend();

Now do the same thing, but compute `w` using only the training data. Then make predictions for both the training and validation data.

In [ ]:
plt.figure()
plt.clf()

Xtrain = xtrain[:,None]
Xvalid = xvalid[:,None]

grid = np.linspace(np.min(x),np.max(x),1000)[:,None]

print("p  |  train err   | valid err |")
print("---+--------------+-----------+")
for p in range(8):
    pf = PolynomialFeatures(p)
    Z = pf.fit_transform(Xtrain)
    ############################################################
    ###                                                      ###
    ###   TODO: solve the normal equations                   ###
    ###                                                      ###
    ############################################################
    w = 0
    grid_poly = pf.fit_transform(grid)
    
    plt.subplot(2,4,p+1)
    plt.scatter(xtrain,ytrain,c='red', label='train',s=5);
#     plt.scatter(xvalid,yvalid,c='green',label='test',s=5);
    plt.plot(grid, grid_poly@w)
    ############################################################
    ###                                                      ###
    ###   TODO: predictions on the training data             ###
    ###                                                      ###
    ############################################################
    train_pred = 0
    train_err = np.sum((ytrain-train_pred)**2)
    Zvalid = pf.transform(Xvalid)
    ############################################################
    ###                                                      ###
    ###   TODO: predictions on the validation data           ###
    ###                                                      ###
    ############################################################
    valid_pred = 0
    valid_err = np.sum((yvalid-valid_pred)**2)
    plt.title('$p=%d$'%p)
    plt.xticks(())
    plt.yticks(())
    plt.ylim((np.min(y),np.max(y)))
    print('%d %12.1f   %12.1f' % (p, train_err, valid_err))
plt.tight_layout();
#     print(max(abs(w)))
#     print(w)

In this case, based on the validation error, a degree of 2 or 3 both seem appropriate. Which is comforting, since we generated the data from a quadratic function ($p=2$). 

## Large weights

Why do the high-degree curves look so wild? Let's look at the weights.

### Fit (interpolate) with a degree 7 polynomial

We have 8 training points, so a degree 7 polynomial can go through every one of them.

Fill in the weights `w` using the normal equations.

In [ ]:
pf = PolynomialFeatures(7)
Z = pf.fit_transform(Xtrain)
############################################################
###                                                      ###
###   TODO: compute w                                    ###
###                                                      ###
############################################################
w = 0

plt.scatter(xtrain,ytrain,c='red', label='train');
plt.scatter(xvalid,yvalid,c='green',label='validation');
plt.plot(grid, pf.transform(grid)@w);
plt.ylim((y.min()-3, y.max()+3))
plt.legend();

In [ ]:
print(w)
print("largest weight:  ", np.max(np.abs(w)))
print("validation error:", np.sum((yvalid - pf.transform(Xvalid)@w)**2))

Look at the weights. The data came from $y = 9x^2$ plus noise, but some weights are far bigger than 9.

Problem: these results are crazy (overfitting). Interpretation: large positive and negative weights are cancelling each other out at the data points.

### Solution 1: lower degree polynomial

Fill in `w` again. What is the shape of `Z` this time?

In [ ]:
pf = PolynomialFeatures(2)
Z = pf.fit_transform(Xtrain)

############################################################
###                                                      ###
###   TODO: compute w                                    ###
###                                                      ###
############################################################
w = 0

plt.scatter(xtrain,ytrain,c='red', label='train');
plt.scatter(xvalid,yvalid,c='green',label='validation');
plt.plot(grid, pf.transform(grid)@w);
plt.ylim((y.min()-3, y.max()+3))
plt.legend();

In [ ]:
print(w)
print("largest weight:  ", np.max(np.abs(w)))
print("validation error:", np.sum((yvalid - pf.transform(Xvalid)@w)**2))

Discussion:
- The lower degree polynomial looks good. 
- But if the true relationship really was complicated? Then if we restricted the degree of the polynomial, we'd miss out on it. 

### Solution 2: add L2-regularization

- Let's keep the high degree, but ask for the weights to be small.
- _We can achieve this by penalizing large values of $w$_.
- The loss becomes

$$\|Zw-y\|^2 + \lambda \|w\|^2$$

With regularization, the normal equations become

$$(Z^TZ + \lambda I)w = Z^Ty$$

Fill in `w`. (`np.eye(k)` is the $k \times k$ identity matrix.)

In [ ]:
pf = PolynomialFeatures(7)
Z = pf.fit_transform(Xtrain)

λ = 0.1
############################################################
###                                                      ###
###   TODO: compute w                                    ###
###                                                      ###
############################################################
w = 0

plt.scatter(xtrain,ytrain,c='red', label='train');
plt.scatter(xvalid,yvalid,c='green',label='validation');
plt.plot(grid, pf.transform(grid)@w);
plt.ylim((y.min()-3, y.max()+3))
plt.legend();

In [ ]:
print(w)
print("largest weight:  ", np.max(np.abs(w)))
print("validation error:", np.sum((yvalid - pf.transform(Xvalid)@w)**2))

Discussion:
- Regularization helped! Even though we used a degree 7 polynomial, we didn't end up with a crazy model.
- Compare the largest weight and the validation error with the degree 7 fit without regularization.
- Degree 2 still does better here, but we only know to pick it because we compared degrees. Regularization works reasonably well without having to guess the right degree.
- We can add regularization to many models, not just least squares with a polynomial basis.